# Load Patterns

The data is clean. Now where does it go?

Loading sounds simple -- just dump it in the database. But "just dump it" causes more production incidents than any other step in the pipeline. Load the same data twice and you get duplicates. Load half the data and the table is in an inconsistent state. Replace the table and you destroy history.

There are three loading strategies, each with different trade-offs. We will implement all three, break one of them on purpose, and then fix it properly.

In [ ]:
%pip install -q sqlalchemy

In [ ]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine, text

## Setting up a test database

We will work with an in-memory SQLite database for experimentation. This means we start fresh every time, which is exactly what we want when testing load strategies.

In [ ]:
# Create some sample cleaned product data
products = pd.DataFrame({
    "product_id": ["P001", "P002", "P003", "P004", "P005"],
    "product_name": [
        "Handmade Ceramic Mug",
        "Vintage Leather Wallet",
        "Artisan Wooden Bowl",
        "Premium Cotton Scarf",
        "Elegant Glass Vase",
    ],
    "price_gbp": [24.99, 34.50, 18.75, 29.99, 42.00],
    "category": ["home", "accessories", "kitchen", "fashion", "decor"],
    "seller": ["alpha", "alpha", "beta", "alpha", "beta"],
})

products

## Strategy 1: Full Refresh

Drop the table and recreate it with the new data. Simple and brutal.

In [ ]:
engine = create_engine("sqlite:///:memory:")

# Load: replace the entire table
products.to_sql("products", engine, if_exists="replace", index=False)

# Verify
with engine.connect() as conn:
    result = pd.read_sql("SELECT * FROM products", conn)
print(f"Table has {len(result)} rows")
result

In [ ]:
# Run it again with slightly different data
updated_products = pd.DataFrame({
    "product_id": ["P001", "P002", "P003", "P006"],
    "product_name": [
        "Handmade Ceramic Mug",
        "Vintage Leather Wallet",
        "Artisan Wooden Bowl",
        "Natural Linen Tablecloth",  # New product
    ],
    "price_gbp": [25.99, 34.50, 18.75, 55.00],  # P001 price changed
    "category": ["home", "accessories", "kitchen", "home"],
    "seller": ["alpha", "alpha", "beta", "beta"],
})

updated_products.to_sql("products", engine, if_exists="replace", index=False)

with engine.connect() as conn:
    result = pd.read_sql("SELECT * FROM products", conn)
print(f"Table has {len(result)} rows")
result

P004 (Premium Cotton Scarf) and P005 (Elegant Glass Vase) are gone. They were in the old data but not in the new data, so the full refresh deleted them.

**Full refresh is appropriate when:**
- You can always rebuild the complete dataset from the source
- The dataset is small enough that loading everything is fast
- You do not need to preserve historical records

**Full refresh is dangerous when:**
- The source data is incomplete (only today's records, not all records)
- Downstream systems depend on the table being complete
- You need to track changes over time

## Strategy 2: Append

Add new rows without touching existing ones.

In [ ]:
engine = create_engine("sqlite:///:memory:")

# First load
products.to_sql("products", engine, if_exists="replace", index=False)

# Second load: append more data
new_products = pd.DataFrame({
    "product_id": ["P006", "P007"],
    "product_name": ["Natural Linen Tablecloth", "Rustic Brass Candleholder"],
    "price_gbp": [55.00, 31.25],
    "category": ["home", "decor"],
    "seller": ["beta", "alpha"],
})

new_products.to_sql("products", engine, if_exists="append", index=False)

with engine.connect() as conn:
    result = pd.read_sql("SELECT * FROM products", conn)
print(f"Table has {len(result)} rows")
result

7 rows. The original 5 are preserved and 2 new ones were added.

But what happens if we accidentally append the same data twice?

In [ ]:
# Oops -- append the same data again
new_products.to_sql("products", engine, if_exists="append", index=False)

with engine.connect() as conn:
    result = pd.read_sql("SELECT * FROM products", conn)
print(f"Table has {len(result)} rows")
result[result["product_id"].isin(["P006", "P007"])]

Duplicates. P006 and P007 each appear twice. The append strategy does not check for existing data -- it blindly adds whatever you give it.

**Append is appropriate when:**
- You are loading genuinely new data (log entries, events, time-series)
- Your extraction is incremental (high-water marks from notebook 1)
- You are confident the same data will never be loaded twice

**Append is dangerous when:**
- Your extraction might produce duplicates
- The pipeline might be retried after a failure
- You do not have a deduplication step downstream

## Strategy 3: Upsert (Insert or Replace)

Insert new rows, update existing ones. The best of both worlds.

SQLite supports `INSERT OR REPLACE`, which works like an upsert: if a row with the same primary key exists, it replaces it. If not, it inserts a new row.

To use this, we need a table with a primary key.

In [ ]:
engine = create_engine("sqlite:///:memory:")

# Create the table with a proper primary key
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE products (
            product_id TEXT PRIMARY KEY,
            product_name TEXT,
            price_gbp REAL,
            category TEXT,
            seller TEXT
        )
    """))
    conn.commit()

print("Table created with PRIMARY KEY on product_id")

In [ ]:
def upsert_products(engine, df):
    """
    Upsert products into the database.
    Inserts new rows, replaces existing ones (matched by product_id).
    """
    with engine.connect() as conn:
        for _, row in df.iterrows():
            conn.execute(text("""
                INSERT OR REPLACE INTO products
                (product_id, product_name, price_gbp, category, seller)
                VALUES (:product_id, :product_name, :price_gbp, :category, :seller)
            """), {
                "product_id": row["product_id"],
                "product_name": row["product_name"],
                "price_gbp": float(row["price_gbp"]),
                "category": row["category"],
                "seller": row["seller"],
            })
        conn.commit()

In [ ]:
# First load
upsert_products(engine, products)

with engine.connect() as conn:
    result = pd.read_sql("SELECT * FROM products", conn)
print(f"After first load: {len(result)} rows")
result

In [ ]:
# Second load: some updated, some new
upsert_batch = pd.DataFrame({
    "product_id": ["P001", "P003", "P006"],
    "product_name": [
        "Handmade Ceramic Mug",       # Same name
        "Artisan Wooden Bowl",         # Same name
        "Natural Linen Tablecloth",    # New product
    ],
    "price_gbp": [25.99, 19.50, 55.00],  # P001 and P003 prices updated
    "category": ["home", "kitchen", "home"],
    "seller": ["alpha", "beta", "beta"],
})

upsert_products(engine, upsert_batch)

with engine.connect() as conn:
    result = pd.read_sql("SELECT * FROM products", conn)
print(f"After second load: {len(result)} rows")
result

6 rows. P001 and P003 were updated with new prices. P006 was inserted. P002, P004, and P005 are untouched. No duplicates, no data loss.

Let's run it again with the same data to prove idempotency.

In [ ]:
# Third load: exact same data as second load
upsert_products(engine, upsert_batch)

with engine.connect() as conn:
    result = pd.read_sql("SELECT * FROM products", conn)
print(f"After third load (same data): {len(result)} rows")

Still 6 rows. Loading the same data again does not create duplicates. This is **idempotent loading** -- the hallmark of a reliable pipeline.

**Upsert is appropriate when:**
- Records have a natural primary key (product ID, order ID)
- You need to update existing records with new values
- Idempotency matters (and it almost always does)

**Upsert is more complex because:**
- You need a primary key (not all data has one)
- Row-by-row iteration is slower than bulk operations
- Different databases have different upsert syntax

## The partial load problem

Here is a scenario that catches people out. You are loading 1,000 rows. At row 500, something goes wrong -- maybe a data type mismatch, maybe a constraint violation, maybe the database runs out of space.

What state is the table in? Let's find out.

In [ ]:
engine = create_engine("sqlite:///:memory:")

# Create table
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE orders (
            order_id INTEGER PRIMARY KEY,
            customer TEXT NOT NULL,
            amount REAL NOT NULL
        )
    """))
    conn.commit()

# Generate 1000 orders, but make row 500 have a problem
orders_data = []
for i in range(1, 1001):
    if i == 500:
        # This row has NULL for customer, which violates NOT NULL
        orders_data.append({"order_id": i, "customer": None, "amount": 99.99})
    else:
        orders_data.append({"order_id": i, "customer": f"Customer {i}", "amount": round(10 + i * 0.5, 2)})

print(f"Prepared {len(orders_data)} orders")
print(f"Row 500 customer: {orders_data[499]['customer']}")

In [ ]:
# Load WITHOUT a transaction -- each insert is auto-committed
loaded = 0
failed = False

with engine.connect() as conn:
    for order in orders_data:
        try:
            conn.execute(text("""
                INSERT INTO orders (order_id, customer, amount)
                VALUES (:order_id, :customer, :amount)
            """), order)
            conn.commit()  # Auto-commit each row
            loaded += 1
        except Exception as e:
            print(f"Error at order {order['order_id']}: {e}")
            failed = True
            break  # Stop on first error

print(f"Loaded: {loaded} rows")
print(f"Failed: {failed}")

# Check the table
with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM orders")).fetchone()[0]
    max_id = conn.execute(text("SELECT MAX(order_id) FROM orders")).fetchone()[0]
print(f"Table has {count} rows, max order_id = {max_id}")

499 rows. The table has orders 1-499, but is missing 500-1000. This is an **inconsistent state**. Half the data is there, half is not.

If a downstream report runs against this table, it will show incorrect totals. If the pipeline is retried, the first 499 rows will fail (duplicate keys) or need to be skipped. It is a mess.

The fix is **transactions**.

## Transactions: all or nothing

A transaction wraps multiple operations into a single atomic unit. Either all of them succeed, or none of them do. If anything goes wrong, the entire batch is rolled back and the table is left untouched.

This is the database equivalent of an undo button.

In [ ]:
engine = create_engine("sqlite:///:memory:")

# Create the table again
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE orders (
            order_id INTEGER PRIMARY KEY,
            customer TEXT NOT NULL,
            amount REAL NOT NULL
        )
    """))
    conn.commit()

# Load WITH a transaction
try:
    with engine.begin() as conn:  # engine.begin() starts a transaction
        for order in orders_data:
            conn.execute(text("""
                INSERT INTO orders (order_id, customer, amount)
                VALUES (:order_id, :customer, :amount)
            """), order)
        # If we get here, all rows inserted successfully -> auto-commit
    print("Transaction committed successfully")
except Exception as e:
    # Any error -> entire transaction rolled back automatically
    print(f"Transaction rolled back: {e}")

# Check the table
with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM orders")).fetchone()[0]
print(f"Table has {count} rows")

Zero rows. The transaction failed at row 500, so the entire batch was rolled back. The table is exactly as it was before we started. Clean. Consistent. No half-loaded garbage.

This is **atomicity** -- one of the four ACID properties of database transactions. It is non-negotiable for production pipelines.

The key difference:
- `engine.connect()` + manual `conn.commit()` -- each statement is independent
- `engine.begin()` -- everything inside the `with` block is a single transaction

## A robust upsert with transactions

In [ ]:
def load_products(engine, df):
    """
    Load products into the database using upsert within a transaction.
    Returns the number of rows loaded.
    """
    with engine.begin() as conn:
        for _, row in df.iterrows():
            conn.execute(text("""
                INSERT OR REPLACE INTO products
                (product_id, product_name, price_gbp, category, seller)
                VALUES (:product_id, :product_name, :price_gbp, :category, :seller)
            """), {
                "product_id": row["product_id"],
                "product_name": row["product_name"],
                "price_gbp": float(row["price_gbp"]),
                "category": row["category"],
                "seller": row["seller"],
            })

    return len(df)

In [ ]:
engine = create_engine("sqlite:///:memory:")

# Create the products table
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE products (
            product_id TEXT PRIMARY KEY,
            product_name TEXT,
            price_gbp REAL,
            category TEXT,
            seller TEXT
        )
    """))
    conn.commit()

In [ ]:
# Load and verify
loaded = load_products(engine, products)
print(f"Loaded {loaded} rows")

with engine.connect() as conn:
    result = pd.read_sql("SELECT * FROM products", conn)
result

## Row count verification

A simple but effective quality check: compare the row count before and after loading.

In [ ]:
def load_with_verification(engine, table_name, df, upsert_fn):
    """
    Load data with before/after row count verification.
    """
    # Count before
    with engine.connect() as conn:
        before_count = conn.execute(
            text(f"SELECT COUNT(*) FROM {table_name}")
        ).fetchone()[0]

    # Load
    loaded = upsert_fn(engine, df)

    # Count after
    with engine.connect() as conn:
        after_count = conn.execute(
            text(f"SELECT COUNT(*) FROM {table_name}")
        ).fetchone()[0]

    print(f"Before: {before_count} rows")
    print(f"Loaded: {loaded} rows from DataFrame")
    print(f"After:  {after_count} rows")
    print(f"Net change: {after_count - before_count:+d} rows")

    return after_count

In [ ]:
# Load the upsert batch (some updates, some new)
upsert_batch = pd.DataFrame({
    "product_id": ["P001", "P003", "P006"],
    "product_name": [
        "Handmade Ceramic Mug",
        "Artisan Wooden Bowl",
        "Natural Linen Tablecloth",
    ],
    "price_gbp": [25.99, 19.50, 55.00],
    "category": ["home", "kitchen", "home"],
    "seller": ["alpha", "beta", "beta"],
})

load_with_verification(engine, "products", upsert_batch, load_products)

We loaded 3 rows but the net change is +1. Two rows were updates (P001, P003) and one was new (P006). The verification tells us exactly what happened.

## Choosing a strategy

| Strategy | Duplicates? | Preserves history? | Idempotent? | Speed |
|----------|------------|-------------------|-------------|-------|
| Full refresh | No | No | Yes | Fast (bulk) |
| Append | Yes (if rerun) | Yes | No | Fast (bulk) |
| Upsert | No | Yes | Yes | Slower (row-by-row) |

In practice:

- **Small reference tables** (sellers, categories): full refresh is fine
- **Event/log data** with reliable incremental extraction: append works
- **Everything else**: upsert. The performance cost is worth the reliability.

You can also combine strategies. Use append for raw staging tables (fast), then upsert from staging into the final analytics table (safe).

## A complete load function

Let's write a production-quality load function that wraps everything we have learned.

In [ ]:
def load_to_database(engine, table_name, df, columns, strategy="upsert"):
    """
    Load a DataFrame into a database table.

    Parameters:
        engine: SQLAlchemy engine
        table_name: Target table name
        df: DataFrame to load
        columns: List of column names (must match table schema)
        strategy: 'replace', 'append', or 'upsert'

    Returns:
        Dict with load statistics
    """
    # Count before
    with engine.connect() as conn:
        try:
            before = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).fetchone()[0]
        except Exception:
            before = 0  # Table might not exist yet

    if strategy == "replace":
        df[columns].to_sql(table_name, engine, if_exists="replace", index=False)

    elif strategy == "append":
        df[columns].to_sql(table_name, engine, if_exists="append", index=False)

    elif strategy == "upsert":
        placeholders = ", ".join(f":{c}" for c in columns)
        col_names = ", ".join(columns)
        sql = f"INSERT OR REPLACE INTO {table_name} ({col_names}) VALUES ({placeholders})"

        with engine.begin() as conn:
            for _, row in df.iterrows():
                params = {c: row[c] for c in columns}
                # Convert numpy types to Python types
                params = {
                    k: (float(v) if isinstance(v, (int, float)) and not isinstance(v, bool)
                        else str(v) if v is not None and not isinstance(v, str)
                        else v)
                    for k, v in params.items()
                }
                conn.execute(text(sql), params)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    # Count after
    with engine.connect() as conn:
        after = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).fetchone()[0]

    return {
        "table": table_name,
        "strategy": strategy,
        "rows_in_df": len(df),
        "rows_before": before,
        "rows_after": after,
        "net_change": after - before,
    }

In [ ]:
# Test it
engine = create_engine("sqlite:///:memory:")

with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE products (
            product_id TEXT PRIMARY KEY,
            product_name TEXT,
            price_gbp REAL,
            category TEXT,
            seller TEXT
        )
    """))
    conn.commit()

cols = ["product_id", "product_name", "price_gbp", "category", "seller"]

stats = load_to_database(engine, "products", products, cols, strategy="upsert")
print(stats)

In [ ]:
# Upsert some updates
stats = load_to_database(engine, "products", upsert_batch, cols, strategy="upsert")
print(stats)

## Summary

What we covered:

- **Full refresh** (`if_exists='replace'`) -- simple but destroys history
- **Append** (`if_exists='append'`) -- preserves history but risks duplicates
- **Upsert** (`INSERT OR REPLACE`) -- best of both, idempotent
- **The partial load problem** -- inserting without a transaction leaves the table inconsistent
- **Transactions** (`engine.begin()`) -- all-or-nothing, the foundation of reliable loading
- **Row count verification** -- simple before/after checks to confirm what happened
- **Choosing a strategy** -- depends on your data, your sources, and your requirements

---

## Exercises

### Exercise 1: Full refresh for sellers

Write a function `refresh_sellers(engine, df)` that does a full refresh of a `sellers` table. The table should have columns: `seller_id`, `name`, `country`, `category`.

Test it by loading data, verifying the count, loading different data, and verifying that the old data is gone.

In [ ]:
# Your code here


### Exercise 2: Safe append

Write a function `safe_append(engine, table_name, df, key_column)` that:

1. Checks which rows in `df` already exist in the table (based on `key_column`)
2. Only appends rows that do not already exist
3. Returns a dict with `inserted` and `skipped` counts

This gives you append behaviour without the duplicate risk.

In [ ]:
# Your code here


### Exercise 3: Transaction rollback

Write a function `load_orders_safe(engine, orders_df)` that:

1. Creates an `orders` table if it does not exist
2. Inserts all orders within a single transaction
3. If any row fails, rolls back the entire transaction
4. Returns `True` if all rows loaded, `False` if the transaction was rolled back

Test it with a batch that contains a bad row and verify the table is empty after the rollback.

In [ ]:
# Your code here


### Exercise 4: Load statistics

Extend the `load_to_database` function to also return:

- `rows_updated`: number of existing rows that were updated (upsert only)
- `rows_inserted`: number of genuinely new rows

Hint: for upsert, `rows_inserted = rows_after - rows_before` and `rows_updated = rows_in_df - rows_inserted`.

In [ ]:
# Your code here
